# Indonesian Spellcheck — Pattern Mining

Companion notebook to `Indonesian Spellcheck Ngram.ipynb`. Mines **frequent
character n-gram patterns** that co-occur in the suspicious words flagged
by the n-gram method.

## What this answers

After the n-gram method flags ~N suspicious tokens, the next question is:
**are these errors random one-offs, or do they cluster around systematic
character patterns?**

Random one-offs (typos) → no pattern. Each error is unique.

Systematic OCR errors (e.g., `str → stgh` substitution applied across many
words) → the trigrams `stgh`, `gh`, `tgh` cooccur in many flagged words.
These patterns are exactly what association rule mining can extract.

## Method

1. **Decompose** each flagged word into its character bigrams + trigrams.
2. Each word becomes a "transaction" (set of n-grams).
3. **Mine frequent itemsets** with FP-Growth — find n-gram combinations
   that appear in many flagged words simultaneously.
4. **Generate association rules** from those itemsets — patterns of the form
   "if word contains `gh`, it likely also contains `tgh`" (this would be
   evidence of the `str→stgh` rule).
5. **Rank** by support (how many flagged words contain this pattern) and
   confidence (how reliably the pattern predicts another part of the
   pattern).

## Output

- `_frequent_itemsets.csv` — n-gram patterns with their support
- `_association_rules.csv` — rules derived from itemsets, with confidence
- `_top_systematic_patterns.csv` — the most common itemsets with example words

## Why FP-Growth instead of Apriori

FP-Growth is more memory-efficient and faster than Apriori for this size of
data (~50–500 flagged words × ~20–50 character n-grams per word). Same
algorithmic family, same outputs (frequent itemsets), just better
implementation. `mlxtend.frequent_patterns.fpgrowth` is a drop-in.

## 1. Configuration

In [17]:
import os
import re
import pandas as pd
from pathlib import Path
from collections import Counter

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

# === Strategy must match the n-gram notebook's run ===
# STRATEGY = "original"
# STRATEGY = "strategy_B"
STRATEGY = "strategy_C"

NGRAM_DIR = Path(f"../Ekstraksi/12. Parallel Corpus - Spellcheck Ngram/{STRATEGY}")
DST_DIR = NGRAM_DIR  # write outputs alongside n-gram outputs

# === Parameters ===
MIN_SUPPORT = 0.05    # itemset must appear in >= 5% of flagged words
MIN_CONFIDENCE = 0.70 # rule confidence threshold
NGRAM_SIZES = [2, 3]  # extract bigrams AND trigrams (gives richer patterns)
MAX_ITEMSET_SIZE = 4  # don't go beyond 4-element itemsets (combinatorial explosion)

print(f"Strategy:         {STRATEGY}")
print(f"N-gram dir:       {NGRAM_DIR}")
print(f"Min support:      {MIN_SUPPORT}")
print(f"Min confidence:   {MIN_CONFIDENCE}")
print(f"Decomposition:    {NGRAM_SIZES}-grams")

assert NGRAM_DIR.exists(), (
    f"Run Indonesian Spellcheck Ngram.ipynb with STRATEGY={STRATEGY} first."
)

top_path = NGRAM_DIR / "_top_suspicious_tokens.csv"
assert top_path.exists(), f"Top suspicious tokens not found: {top_path}"

Strategy:         strategy_C
N-gram dir:       ..\Ekstraksi\12. Parallel Corpus - Spellcheck Ngram\strategy_C
Min support:      0.05
Min confidence:   0.7
Decomposition:    [2, 3]-grams


## 2. Load flagged words

In [18]:
flagged_df = pd.read_csv(top_path)
print(f"Loaded {len(flagged_df):,} flagged tokens")
print(f"\nTop 10 sample:")
print(flagged_df.head(10).to_string(index=False))

# Use all flagged tokens (not just top 200) for richer pattern mining.
# But weight each by occurrence count when computing support.
flagged_tokens = flagged_df["token"].tolist()
print(f"\nUsing all {len(flagged_tokens)} unique flagged tokens for mining")

Loaded 200 flagged tokens

Top 10 sample:
token  occurrences  perplexity
  utk         1202        16.7
  anu         1002        13.2
  meu          696        39.5
   vt          627        12.7
  org          530        21.3
jeung          522        19.8
   km          509        15.3
  rsd          486       389.1
 jite          430        14.5
   sj          393        20.5

Using all 200 unique flagged tokens for mining


## 3. Decompose words into n-gram transactions

Each flagged word becomes a "transaction": a set of bigrams and trigrams it
contains. Boundary characters (`^`, `$`) are preserved so we can mine
patterns that depend on word position.

In [19]:
PADDING_CHAR = "^"
END_CHAR = "$"


def word_to_features(word: str, n_sizes=NGRAM_SIZES) -> set:
    """Decompose word into the union of its n-grams for each n."""
    features = set()
    for n in n_sizes:
        # Pad based on the n-gram size
        padded = (PADDING_CHAR * (n - 1)) + word + (END_CHAR * (n - 1))
        for i in range(len(padded) - n + 1):
            ngram = padded[i : i + n]
            features.add(f"{n}g:{ngram}")
    return features


# Sanity check
print(f"Sample features for 'ekstghak':")
for f in sorted(word_to_features("ekstghak")):
    print(f"  {f}")

Sample features for 'ekstghak':
  2g:^e
  2g:ak
  2g:ek
  2g:gh
  2g:ha
  2g:k$
  2g:ks
  2g:st
  2g:tg
  3g:^^e
  3g:^ek
  3g:ak$
  3g:eks
  3g:gha
  3g:hak
  3g:k$$
  3g:kst
  3g:stg
  3g:tgh


## 4. Build the transaction matrix

Each row is a flagged word; each column is a possible n-gram feature.
Cell is True if the word contains that feature, False otherwise.

In [20]:
transactions = [word_to_features(word) for word in flagged_tokens]
print(f"Built {len(transactions)} transactions")
print(f"Average features per transaction: "
      f"{sum(len(t) for t in transactions) / max(1, len(transactions)):.1f}")

# Encode for mlxtend
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_array, columns=te.columns_)
print(f"Transaction matrix shape: {df_trans.shape}")
print(f"  ({df_trans.shape[0]} words × {df_trans.shape[1]} unique n-gram features)")

Built 200 transactions
Average features per transaction: 11.0
Transaction matrix shape: (200, 899)
  (200 words × 899 unique n-gram features)


## 5. Mine frequent itemsets with FP-Growth

In [21]:
print(f"Mining itemsets with min_support={MIN_SUPPORT}, max_len={MAX_ITEMSET_SIZE}...")

itemsets = fpgrowth(
    df_trans,
    min_support=MIN_SUPPORT,
    max_len=MAX_ITEMSET_SIZE,
    use_colnames=True,
)

# Sort by support and itemset size
itemsets["itemset_size"] = itemsets["itemsets"].apply(len)
itemsets = itemsets.sort_values(
    ["itemset_size", "support"], ascending=[True, False]
).reset_index(drop=True)

# Convert frozenset to sorted string for readability
itemsets["pattern"] = itemsets["itemsets"].apply(
    lambda s: " + ".join(sorted(s))
)

print(f"\nFound {len(itemsets):,} frequent itemsets")
print(f"By size:")
print(itemsets["itemset_size"].value_counts().sort_index())

# Save
itemsets[["pattern", "support", "itemset_size"]].to_csv(
    DST_DIR / "_frequent_itemsets.csv", index=False
)

print(f"\n=== Top 20 frequent itemsets (size >= 2) ===")
top_2plus = itemsets[itemsets["itemset_size"] >= 2].head(20)
print(top_2plus[["pattern", "support", "itemset_size"]].to_string(index=False))

Mining itemsets with min_support=0.05, max_len=4...

Found 47 frequent itemsets
By size:
itemset_size
1    33
2    14
Name: count, dtype: int64

=== Top 20 frequent itemsets (size >= 2) ===
       pattern  support  itemset_size
2g:o$ + 3g:o$$    0.150             2
2g:e$ + 3g:e$$    0.135             2
2g:a$ + 3g:a$$    0.120             2
2g:u$ + 3g:u$$    0.115             2
2g:^a + 3g:^^a    0.110             2
2g:^m + 3g:^^m    0.105             2
2g:i$ + 3g:i$$    0.080             2
2g:^n + 3g:^^n    0.080             2
2g:^b + 3g:^^b    0.075             2
2g:^i + 3g:^^i    0.065             2
2g:^t + 3g:^^t    0.060             2
2g:g$ + 3g:g$$    0.055             2
2g:^s + 3g:^^s    0.050             2
2g:^e + 3g:^^e    0.050             2


## 6. Generate association rules

Rules of the form "if word contains pattern A, then it likely contains
pattern B" — quantified with support, confidence, and lift.

- **Support**: fraction of flagged words containing both A and B
- **Confidence**: P(B | A) — when A appears, how often does B?
- **Lift**: how much more likely is B given A vs. P(B) alone (>1 means
  positive association)

In [22]:
if len(itemsets[itemsets["itemset_size"] >= 2]) > 0:
    rules = association_rules(
        itemsets[["itemsets", "support"]],
        metric="confidence",
        min_threshold=MIN_CONFIDENCE,
        num_itemsets=len(df_trans),
    )

    # Format antecedents/consequents for readability
    rules["antecedents_str"] = rules["antecedents"].apply(lambda s: " + ".join(sorted(s)))
    rules["consequents_str"] = rules["consequents"].apply(lambda s: " + ".join(sorted(s)))

    # Sort by lift (most surprising rules first), then confidence
    rules = rules.sort_values(["lift", "confidence"], ascending=[False, False]).reset_index(drop=True)

    out = rules[["antecedents_str", "consequents_str", "support",
                 "confidence", "lift"]].copy()
    out.columns = ["antecedent", "consequent", "support", "confidence", "lift"]
    out.to_csv(DST_DIR / "_association_rules.csv", index=False)

    print(f"Generated {len(rules):,} rules with confidence >= {MIN_CONFIDENCE}")
    print(f"\n=== Top 20 rules by lift ===")
    print(out.head(20).to_string(index=False))
else:
    print("No itemsets of size >= 2 found. Try lowering MIN_SUPPORT.")
    rules = pd.DataFrame()

Generated 28 rules with confidence >= 0.7

=== Top 20 rules by lift ===
antecedent consequent  support  confidence      lift
    3g:^^s      2g:^s    0.050         1.0 20.000000
     2g:^s     3g:^^s    0.050         1.0 20.000000
    3g:^^e      2g:^e    0.050         1.0 20.000000
     2g:^e     3g:^^e    0.050         1.0 20.000000
     2g:g$     3g:g$$    0.055         1.0 18.181818
    3g:g$$      2g:g$    0.055         1.0 18.181818
     2g:^t     3g:^^t    0.060         1.0 16.666667
    3g:^^t      2g:^t    0.060         1.0 16.666667
     2g:^i     3g:^^i    0.065         1.0 15.384615
    3g:^^i      2g:^i    0.065         1.0 15.384615
     2g:^b     3g:^^b    0.075         1.0 13.333333
    3g:^^b      2g:^b    0.075         1.0 13.333333
    3g:i$$      2g:i$    0.080         1.0 12.500000
     2g:i$     3g:i$$    0.080         1.0 12.500000
     2g:^n     3g:^^n    0.080         1.0 12.500000
    3g:^^n      2g:^n    0.080         1.0 12.500000
     2g:^m     3g:^^m    0.

## 7. Group itemsets by example words

For each frequent itemset, find example flagged words that contain all its
features. This makes the patterns concrete — instead of just "the trigrams
`gh` and `tgh` cooccur," we show "they cooccur in `petigh`, `tughun`, ..."

In [23]:
# For each itemset (size >= 2), list example words that match all its features
def find_examples(itemset_features, words, max_examples=8) -> list:
    matched = []
    for word in words:
        word_features = word_to_features(word)
        if itemset_features.issubset(word_features):
            matched.append(word)
            if len(matched) >= max_examples:
                break
    return matched


multi_itemsets = itemsets[itemsets["itemset_size"] >= 2].head(30).copy()
multi_itemsets["examples"] = multi_itemsets["itemsets"].apply(
    lambda s: "; ".join(find_examples(s, flagged_tokens))
)

systematic_path = DST_DIR / "_top_systematic_patterns.csv"
multi_itemsets[["pattern", "support", "itemset_size", "examples"]].to_csv(
    systematic_path, index=False
)

print(f"=== Top 15 systematic patterns with example words ===\n")
for _, row in multi_itemsets.head(15).iterrows():
    print(f"Pattern: {row['pattern']}")
    print(f"  Support:  {row['support']:.3f} ({int(row['support'] * len(flagged_tokens))} of {len(flagged_tokens)} flagged words)")
    print(f"  Examples: {row['examples']}")
    print()

print(f"Saved to: {systematic_path}")

=== Top 15 systematic patterns with example words ===

Pattern: 2g:o$ + 3g:o$$
  Support:  0.150 (30 of 200 flagged words)
  Examples: ltuyuo; tiyo; wo; ito; iko; boyito; aluo; apo

Pattern: 2g:e$ + 3g:e$$
  Support:  0.135 (27 of 200 flagged words)
  Examples: jite; iye; ate; ode; ike; nee; ane; erbage

Pattern: 2g:a$ + 3g:a$$
  Support:  0.120 (24 of 200 flagged words)
  Examples: kiota; hna; dma; haalia; manehna; ginaa; noota; maroa

Pattern: 2g:u$ + 3g:u$$
  Support:  0.115 (23 of 200 flagged words)
  Examples: anu; meu; wau; wonu; etu; henteu; waqu; wagu

Pattern: 2g:^a + 3g:^^a
  Support:  0.110 (22 of 200 flagged words)
  Examples: anu; ate; ahli; ane; aluo; apo; are; ago

Pattern: 2g:^m + 3g:^^m
  Support:  0.105 (21 of 200 flagged words)
  Examples: meu; ml; mowali; manehna; msl; mohuo; moali; muhammad

Pattern: 2g:i$ + 3g:i$$
  Support:  0.080 (16 of 200 flagged words)
  Examples: pasteyi; ahli; mowali; yi; peyi; doi; ei; moali

Pattern: 2g:^n + 3g:^^n
  Support:  0.080 (16 o

## 8. Aggregate stats

For thesis tables and side-by-side strategy comparison.

In [24]:
aggregate = {
    "method": "patterns",
    "strategy": STRATEGY,
    "n_flagged_tokens_input": len(flagged_tokens),
    "n_unique_features": df_trans.shape[1],
    "n_frequent_itemsets":    len(itemsets),
    "n_itemsets_size_2plus":  int((itemsets["itemset_size"] >= 2).sum()),
    "n_itemsets_size_3plus":  int((itemsets["itemset_size"] >= 3).sum()),
    "n_association_rules":    len(rules) if len(rules) > 0 else 0,
    "min_support_used":     MIN_SUPPORT,
    "min_confidence_used":  MIN_CONFIDENCE,
    "max_itemset_size":     MAX_ITEMSET_SIZE,
}

agg_df = pd.DataFrame([aggregate])
agg_df.to_csv(DST_DIR / "_patterns_aggregate.csv", index=False)
for k, v in aggregate.items():
    print(f"  {k:<28} {v}")
print(f"\nAll pattern outputs written to: {DST_DIR.resolve()}")

  method                       patterns
  strategy                     strategy_C
  n_flagged_tokens_input       200
  n_unique_features            899
  n_frequent_itemsets          47
  n_itemsets_size_2plus        14
  n_itemsets_size_3plus        0
  n_association_rules          28
  min_support_used             0.05
  min_confidence_used          0.7
  max_itemset_size             4

All pattern outputs written to: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\12. Parallel Corpus - Spellcheck Ngram\strategy_C


## 9. Interpretation guide

**Reading the systematic patterns output:**

A pattern like `2g:gh + 3g:gh$` with support=0.4 means: 40% of flagged words
end in `gh`. Combined with examples like `petigh`, `tughun`, `pegh`, this
points to a systematic OCR substitution where Indonesian word-final `r`
got rendered as `gh` (a known Jambi-dialect rendering issue).

**Reading the association rules:**

A rule like `2g:gh → 3g:tgh` with confidence=0.85 means: when a flagged
word contains `gh`, 85% of the time it also contains `tgh` — pointing to
the more specific `str → stgh` substitution that produces `stgh` clusters.

**What to do with these patterns:**

The systematic patterns identify **upstream extraction bugs that explain
multiple errors at once**. Each pattern with high support is a candidate
for a targeted upstream fix. For example, if you find that `str → stgh`
explains 60% of flagged words, fixing that one OCR substitution rule at
the extraction stage would clean up 60% of the flagged-word population.

**Limitations:**

- Patterns reflect what flagged words have in common — they don't tell us
  whether the flagging itself was correct (need gold annotation for that).
- Highly common Indonesian patterns (e.g., `2g:an`, the ending of many
  Indonesian words) appear in many flagged words by chance. Filter for
  unusual-looking patterns when interpreting.
- Pattern mining doesn't distinguish "bug" from "loanword convention" —
  e.g., the pattern `q + u` would appear in `quran`, `quota` etc. (real)
  not just in OCR garbage.